In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATA_PATH = "/content/drive/MyDrive/cs639_guardrails/data/sampled_5k.csv"
EVAL_PATH = "/content/drive/MyDrive/cs639_guardrails/data/eval_500.csv"
SAVE_DIR  = "/content/drive/MyDrive/cs639_guardrails/s3_activations"

import pandas as pd

Train: 5000, Eval: 500, Overlap: 0
Eval label distribution: {0: 250, 1: 250}


In [3]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [4]:
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df)}")
print(f"Label distribution: {df['off_topic'].value_counts().to_dict()}")
print(f"Unique system prompts: {df['system_prompt'].nunique()}")
print(f"Avg prompt word count: {df['prompt_word_count'].mean():.1f}")

Rows: 5000
Label distribution: {0: 2500, 1: 2500}
Unique system prompts: 500
Avg prompt word count: 8.6


In [5]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
NUM_LAYERS = model.config.num_hidden_layers  # 32 for Llama-3.1-8B
HIDDEN_DIM = model.config.hidden_size          # 4096
print(f"Layers: {NUM_LAYERS}, hidden dim: {HIDDEN_DIM}")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Layers: 32, hidden dim: 4096


In [10]:
activations = {}

def make_hook(layer_idx):
    def hook_fn(module, inp, out):
        # In transformers v5, LlamaDecoderLayer returns hidden_states directly
        # Shape: (batch, seq_len, hidden_dim)
        hidden = out[0] if isinstance(out, tuple) else out

        if hidden.dim() != 3:
            raise ValueError(f"Layer {layer_idx}: expected 3D, got {hidden.shape}")

        # Grab last token's representation
        activations[layer_idx] = hidden[:, -1, :].detach().cpu().float()
    return hook_fn

In [12]:
# === Step 1: Remove all existing hooks ===
for h in hooks:
    h.remove()
hooks = []
activations = {}

# === Step 2: Redefine make_hook with the fix ===
def make_hook(layer_idx):
    def hook_fn(module, inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        if hidden.dim() != 3:
            raise ValueError(f"Layer {layer_idx}: shape {hidden.shape}")
        activations[layer_idx] = hidden[:, -1, :].detach().cpu().float()
    return hook_fn

# === Step 3: Re-register ===
for i, layer in enumerate(model.model.layers):
    hooks.append(layer.register_forward_hook(make_hook(i)))

print(f"Registered {len(hooks)} hooks")

# === Step 4: Test on one example ===
row = df.iloc[0]
messages = [
    {"role": "system", "content": row["system_prompt"]},
    {"role": "user", "content": row["prompt"]}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

with torch.no_grad():
    model(**inputs)

print(f"Captured {len(activations)} layers")
print(f"Layer 0 shape: {activations[0].shape}")

Registered 32 hooks
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 207, 4096])
Captured 32 layers
Layer 0 shape: torch.Size([1, 4096])


In [13]:
# === Cell 6: Full extraction loop ===
all_acts = {i: [] for i in range(NUM_LAYERS)}
all_labels = []
all_indices = []

from tqdm import tqdm
for idx, row in tqdm(df.iterrows(), total=len(df)):
    messages = [
        {"role": "system", "content": row["system_prompt"]},
        {"role": "user", "content": row["prompt"]}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    with torch.no_grad():
        model(**inputs)

    for layer_idx in range(NUM_LAYERS):
        all_acts[layer_idx].append(activations[layer_idx].numpy())
    all_labels.append(int(row["off_topic"]))
    all_indices.append(int(row["__index_level_0__"]))


  0%|          | 2/5000 [00:00<06:46, 12.28it/s]

out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 207, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 125, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 150, 4096])



  0%|          | 6/5000 [00:00<05:41, 14.63it/s]

out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 178, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 222, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 127, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 160, 4096])



  0%|          | 10/5000 [00:00<05:14, 15.89it/s]

...


100%|█████████▉| 4994/5000 [04:46<00:00, 17.46it/s]

out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 172, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 114, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 112, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 175, 4096])



100%|█████████▉| 4998/5000 [04:47<00:00, 17.39it/s]

out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 126, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 221, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 231, 4096])
out type: <class 'torch.Tensor'>
out shape: torch.Size([1, 171, 4096])



100%|██████████| 5000/5000 [04:47<00:00, 17.40it/s]


In [14]:
# === Cell 7: Save ===
labels_arr = np.array(all_labels)
indices_arr = np.array(all_indices)
np.save(f"{SAVE_DIR}/labels.npy", labels_arr)
np.save(f"{SAVE_DIR}/indices.npy", indices_arr)

for layer_idx in range(NUM_LAYERS):
    stacked = np.vstack(all_acts[layer_idx])  # (5000, 4096)
    np.save(f"{SAVE_DIR}/layer_{layer_idx:02d}.npy", stacked)

print(f"Saved {NUM_LAYERS} layer files + labels to {SAVE_DIR}")
print(f"Per-layer file size: ~{5000 * 4096 * 4 / 1e6:.1f} MB")

Saved 32 layer files + labels to /content/drive/MyDrive/cs639_guardrails/s3_activations
Per-layer file size: ~81.9 MB
